# 08A Deep Learning with PyTorch (Lightning)

`pip install torch`  
`python3 -m pip install -U torch --user`

`pip install lightning`  
`python3 -m pip install -U lightning --user`

`pip install torchvision`  
`python3 -m pip install -U torchvision --user`

In [1]:
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import multiprocessing

import numpy as np
import lightning.pytorch as pl
import torch
import torchmetrics
import torchvision
import torch.nn as nn
import torch.optim as optim

# To plot pretty figures
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [2]:
# Check if Metal Performance Shaders is available - on macOS
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal Performance Shaders) device.")
    
# Check for CUDA (NVIDIA GPU)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Properties: {torch.cuda.get_device_properties(0)}")
    
# Default to CPU if no accelerators are available
else:
    device = torch.device("cpu")
    print("Using CPU device.")

# Display summary of available devices
print(f"Selected device: {device}")
print("Available devices summary:")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"MPS Available: {torch.backends.mps.is_available()}")

Using CPU device.
Selected device: cpu
Available devices summary:
CUDA Available: False
MPS Available: False


In [3]:
num_cores = multiprocessing.cpu_count()
print(f"My machine has {num_cores} available cores")

My machine has 8 available cores


## PyTorch Tensors

Create tensors from data:

In [4]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)
x_data

tensor([[1, 2],
        [3, 4]])

Create tensors from numpy arrays:

In [5]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np

tensor([[1, 2],
        [3, 4]], dtype=torch.int32)

And you can go back to NumPy:

In [6]:
np_array_2 = x_np.numpy()
type(np_array_2)

numpy.ndarray

You'll find many mehods and functions with similarities to NumPy

In [7]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.9997, 0.7299],
        [0.6330, 0.3167]]) 

Random Tensor: 
 tensor([[0.6886, 0.2433, 0.8929],
        [0.5840, 0.8513, 0.7901]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


Attributes of a tensor:

In [5]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


Arithmetic operations on tensors:

In [9]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)

tensor([[0.9503, 0.4999, 0.9901],
        [0.4999, 0.7257, 0.9703],
        [0.9901, 0.9703, 1.6070]])

In [7]:
# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[0.1805, 0.0121, 0.2800, 0.9046],
        [0.1710, 0.0366, 0.0468, 0.8999],
        [0.9840, 0.7818, 0.8958, 0.4407]])

## Download training and test set into pyTorch datasets

In [5]:
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms

train_val_set = torchvision.datasets.FashionMNIST(
    root="../datasets",
    train=True,
    download=True,
#    transform=ToTensor()
    transform=transforms.Compose(
        [transforms.Resize(16), transforms.ToTensor()]
    )
)

test_set = torchvision.datasets.FashionMNIST(
    root="../datasets",
    train=False,
    download=True,
    transform=ToTensor()
)

In [ ]:
import matplotlib.pyplot as plt
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}
figure = plt.figure(figsize=(4, 4))
cols, rows = 1, 1
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_val_set), size=(1,)).item()
    img, label = train_val_set[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()


<img src="clothes.png"> 


In [26]:
from torch.utils.data import Dataset
isinstance(train_val_set, Dataset)

True

In [27]:
len(train_val_set)

60000

In [8]:
len(test_set)

10000

A bit like for the standard MNIST dataset we got 60,000 samples in the training/validation set and 10,000 in the test set.

Let's create a validation set with 6,000 samples (i.e. 10%) out of `train_val_set`

In [9]:
from torch.utils.data.sampler import SubsetRandomSampler

def train_val_split(
    train_valid_set: Dataset,
    val_size: float | int = 0.2,
    shuffle: bool = False,
    random_seed: int = 0,
) -> tuple[SubsetRandomSampler, SubsetRandomSampler]:
    """
    Retuns two SubsetRandomSampler to split the full training set into a training and a
    validation part
    """
    num_train_val = len(train_valid_set)
    indices = list(range(num_train_val))
    split = round(val_size * num_train_val) if type(val_size) == float else val_size
    if shuffle is True:
        np.random.seed(random_seed)
        np.random.shuffle(indices)
    train_idx, valid_idx = indices[split:], indices[:split]
    print(f"train_idx len = {len(train_idx)}; valid_idx len = {len(valid_idx)}")
    train_sampler = SubsetRandomSampler(train_idx)
    val_sampler = SubsetRandomSampler(valid_idx)
    return train_sampler, val_sampler

In [10]:
train_sampler, val_sampler = train_val_split(train_val_set)

train_idx len = 48000; valid_idx len = 12000


We now have our data loaded as a PyTorch `Dataset`. There is one more step we need to feed the data into a PyTorch/PyTorch Lightning model, which is instantiating a `DataLoader`. This
will pass the data in batches to the model.

In [11]:
from torch.utils.data import DataLoader

num_workers = num_cores - 2
batch_size = 128

train_loader = DataLoader(
    dataset=train_val_set,
    batch_size=batch_size, 
    sampler=train_sampler,
    num_workers=num_workers,
    pin_memory=True,
)
valid_loader = DataLoader(
    dataset=train_val_set,
    batch_size=batch_size,
    sampler=val_sampler,
    num_workers=num_workers,
    pin_memory=True,
)

**Exercise:** write a data_loader for the test set.

In [12]:
# write your solution here
test_loader = DataLoader(
    dataset=test_set,
    batch_size=batch_size, 
    num_workers=num_workers,
    pin_memory=True,
)

In [13]:
len(train_loader)

375

In [14]:
len(train_loader.dataset)

60000

In [15]:
train_loader.batch_size * len(train_loader)

48000

## Define a Fully connected network

Next step we will build a network and we will define the training loop.

**Exercise 1:** complete the sequential model below. You will have to add two hidden `Linear` layers (with 128 and 256 neurons respectively and a suitable activation function) and one `Linear` output layer with an appropriare number of output nodes and an appropriate activation function. 


In [16]:
class SimpleMNISTClassifier(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(28 * 28, 128)
        self.layer_2 = nn.Linear(128, 256)
        self.layer_3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)  # apply softmax to the input dimension

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for our Classifier
        """
        batch_size, channels, height, width = x.size()
        # print(f"Batch size = {batch_size}, channels = {channels}, width = {width}, height = {height}")
        # flatten the input image across (channels, height, width)
        x = x.view(batch_size, -1)
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        x = self.softmax(x)
        return x


    def training_step(self, batch, batch_idx):
        x, y = batch
        # print(f"Batch idx #{batch_idx}. Input shape = {x.shape}; Output shape = {y.shape}")
        # batch_size, channels, height, width = x.size()
        # print(f"Batch size = {batch_size}, channels = {channels}, width = {width}, height = {height}")
        logits = self.forward(x)
        loss = nn.functional.cross_entropy(logits, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch
        val_logits = self.forward(x)
        val_loss = nn.functional.cross_entropy(val_logits, y)
        self.log("val_loss", val_loss, on_epoch=True)
        val_accuracy = torchmetrics.functional.accuracy(
            val_logits,
            y,
            task="multiclass",
            num_classes=10,
        )
        self.log("val_accuracy", val_accuracy, on_step=True, on_epoch=True)

    def test_step(self):
        pass

    def prediction_step(self):
        pass

    def configure_optimizers(self):
        optimizer = optim.SGD(self.parameters(), lr=1e-2)
        return optimizer

In [17]:
from lightning.pytorch.loggers import MLFlowLogger
logger = MLFlowLogger(experiment_name="fashion_mnist", tracking_uri="file:./mlruns")

In [18]:
from lightning import Trainer
simple_classifier = SimpleMNISTClassifier()
trainer = Trainer(
    accelerator="cpu",
    deterministic=True,
    max_epochs=10,
    logger=logger,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


The `Trainer` object is where most of the "magic" happens. Under the hood it handles all the loop details that you would have to write yourself if you were writing vanilla PyTorch.

You can train it in a way very similar to how you trained your scikit-learn models.

In [19]:
trainer.fit(
    model=simple_classifier,
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader,
)

Experiment with name fashion_mnist not found. Creating it.

  | Name    | Type    | Params | Mode 
--------------------------------------------
0 | layer_1 | Linear  | 100 K  | train
1 | layer_2 | Linear  | 33.0 K | train
2 | layer_3 | Linear  | 2.6 K  | train
3 | relu    | ReLU    | 0      | train
4 | softmax | Softmax | 0      | train
--------------------------------------------
136 K     Trainable params
0         Non-trainable params
136 K     Total params
0.544     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\giselara\anaconda3\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x256 and 784x128)

In [20]:
!mlflow ui

^C
